In [6]:
import pandas as pd
import re

# Load the dataset
data = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
aspect_keywords = {
    'Action': ['action', 'fight', 'battle', 'thrill', 'adventure', 'chase', 'explosive', 'combat'],
    'Cinematography': ['cinematography', 'visuals', 'camera', 'shots', 'photography', 'scenery', 'visual effects'],
    'Acting': ['acting', 'performance', 'actor', 'actress', 'role', 'portrayal', 'cast'],
    'Plot': ['plot', 'story', 'narrative', 'script', 'writing', 'storyline'],
    'Sound': ['sound', 'music', 'score', 'audio', 'effects', 'soundtrack']
}

# Function to analyze sentiment using a more advanced method
from textblob import TextBlob

def analyze_sentiment(text):
    analysis = TextBlob(text)
    # Classify based on polarity
    if analysis.sentiment.polarity > 0:
        return 'positive'
    elif analysis.sentiment.polarity < 0:
        return 'negative'
    else:
        return 'neutral'

# Modified extract_aspect_sentiment function
def extract_aspect_sentiment(review):
    review_cleaned = clean_text(review)
    aspect_sentiment = {}
    
    for aspect, keywords in aspect_keywords.items():
        if any(keyword in review_cleaned for keyword in keywords):
            sentiment = analyze_sentiment(review)  # Use TextBlob for sentiment
            aspect_sentiment[aspect] = sentiment

    return aspect_sentiment

# Apply the updated function to each review
data['aspect_sentiments'] = data['review'].apply(extract_aspect_sentiment)

# Display the first few results
print(data[['review', 'aspect_sentiments']].head())



                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                   aspect_sentiments  
0                                                 {}  
1                             {'Acting': 'positive'}  
2                               {'Plot': 'positive'}  
3  {'Action': 'negative', 'Cinematography': 'nega...  
4       {'Action': 'positive', 'Acting': 'positive'}  


In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
rows = []
for index, row in data.iterrows():
    for aspect, sentiment in row['aspect_sentiments'].items():
        rows.append({'aspect': aspect, 'sentiment': sentiment, 'review': row['review']})

# Convert to DataFrame
aspect_data = pd.DataFrame(rows)

# Encode sentiment labels
label_encoder = LabelEncoder()  # Ensure this is defined
aspect_data['sentiment_encoded'] = label_encoder.fit_transform(aspect_data['sentiment'])

# Split into features and labels
X_aspect = aspect_data['review']
y_aspect = aspect_data['sentiment_encoded']

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Vectorize the text data
vectorizer = TfidfVectorizer(max_features=5000)
X_aspect_tfidf = vectorizer.fit_transform(X_aspect).toarray()

# Split the data into training and testing sets
X_train_aspect, X_test_aspect, y_train_aspect, y_test_aspect = train_test_split(
    X_aspect_tfidf, y_aspect, test_size=0.2, random_state=42
)


In [15]:
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models

# Build the model for aspect sentiment analysis
model_aspect = models.Sequential([
    layers.Input(shape=(X_train_aspect.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(label_encoder.classes_), activation='softmax')  # Use softmax for multi-class
])

# Compile the model
model_aspect.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
history_aspect = model_aspect.fit(X_train_aspect, y_train_aspect, epochs=10, batch_size=16, 
                                   validation_split=0.2)

y_pred_aspect = model_aspect.predict(X_test_aspect)
y_pred_aspect_classes = y_pred_aspect.argmax(axis=-1)  # Get the predicted classes

# Calculate and print accuracy
accuracy = accuracy_score(y_test_aspect, y_pred_aspect_classes)
print(f'Accuracy: {accuracy * 100:.2f}%')



Epoch 1/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.8598 - loss: 0.3455 - val_accuracy: 0.9409 - val_loss: 0.1451
Epoch 2/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.9525 - loss: 0.1246 - val_accuracy: 0.9565 - val_loss: 0.1189
Epoch 3/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9756 - loss: 0.0697 - val_accuracy: 0.9650 - val_loss: 0.1073
Epoch 4/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.9854 - loss: 0.0437 - val_accuracy: 0.9685 - val_loss: 0.1032
Epoch 5/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9886 - loss: 0.0360 - val_accuracy: 0.9672 - val_loss: 0.1262
Epoch 6/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9915 - loss: 0.0267 - val_accuracy: 0.9683 - val_loss: 0.1336
Epoch 7/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9920 - loss: 0.0252 - val_accuracy: 0.9679 - val_loss: 0.1405
Epoch 8/10
3611/3611 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9939 - loss: 0